# 04 — Twitch Neural Ranker

Train a compact PyTorch MLP ranker on the engineered Twitch features.

The model remains CPU-friendly and is deliberately small enough to become a realistic serving workload later. We save:
- preprocessing statistics
- trained PyTorch weights
- test inference scores
- training history

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from time import perf_counter
import json
import joblib

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score

import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

PROJECT_DIR = Path("Distributed ML Inference & Ranking")
DATA_DIR = PROJECT_DIR / "data/processed/twitch"
ARTIFACT_DIR = PROJECT_DIR / "artifacts/twitch"
PRED_DIR = PROJECT_DIR / "data/processed/twitch/predictions"

ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
PRED_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = DATA_DIR / "twitch_train_ranking.csv.gz"
TEST_PATH = DATA_DIR / "twitch_test_ranking.csv.gz"

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# Keep the main benchmark CPU-first.
DEVICE = torch.device("cpu")
print("Device:", DEVICE)

## 1. Load data and define features

In [ ]:
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

ID_COLUMNS = ["user_id", "streamer"]
LABEL_COLUMN = "label"
FUTURE_ONLY_COLUMNS = ["future_interactions", "future_watch_minutes"]

FEATURE_COLUMNS = [
    c for c in train.columns
    if c not in ID_COLUMNS + [LABEL_COLUMN] + FUTURE_ONLY_COLUMNS
]

print("Train:", train.shape)
print("Test: ", test.shape)
print("Features:", len(FEATURE_COLUMNS))

## 2. Fit preprocessing on training data only

In [ ]:
imputer = SimpleImputer(strategy="median")
scaler = StandardScaler()

X_train = imputer.fit_transform(train[FEATURE_COLUMNS])
X_train = scaler.fit_transform(X_train)

X_test = imputer.transform(test[FEATURE_COLUMNS])
X_test = scaler.transform(X_test)

y_train = train["label"].to_numpy(dtype=np.float32)
y_test = test["label"].to_numpy(dtype=np.float32)

X_train = X_train.astype(np.float32)
X_test = X_test.astype(np.float32)

print(X_train.shape, X_test.shape)

## 3. Create PyTorch data loaders

In [ ]:
BATCH_SIZE = 2048

train_ds = TensorDataset(
    torch.from_numpy(X_train),
    torch.from_numpy(y_train),
)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
)

print(f"Training batches: {len(train_loader):,}")

## 4. Define neural ranking model

In [ ]:
class TwitchMLPRanker(nn.Module):
    def __init__(self, input_dim):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.10),

            nn.Linear(64, 32),
            nn.ReLU(),

            nn.Linear(32, 1),
        )

    def forward(self, x):
        return self.network(x).squeeze(-1)


model = TwitchMLPRanker(
    input_dim=len(FEATURE_COLUMNS)
).to(DEVICE)

model

## 5. Train

In [ ]:
EPOCHS = 4
LEARNING_RATE = 1e-3

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=1e-4,
)

loss_fn = nn.BCEWithLogitsLoss()

history = []

for epoch in range(1, EPOCHS + 1):
    model.train()

    running_loss = 0.0
    n_examples = 0

    start = perf_counter()

    for features, labels in train_loader:
        features = features.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()

        logits = model(features)
        loss = loss_fn(logits, labels)

        loss.backward()
        optimizer.step()

        batch_n = len(labels)
        running_loss += loss.item() * batch_n
        n_examples += batch_n

    epoch_loss = running_loss / n_examples
    elapsed = perf_counter() - start

    history.append(
        {
            "epoch": epoch,
            "train_loss": epoch_loss,
            "seconds": elapsed,
        }
    )

    print(
        f"Epoch {epoch}/{EPOCHS} | "
        f"loss={epoch_loss:.5f} | "
        f"time={elapsed:.2f}s"
    )

## 6. Plot training loss

In [ ]:
history_df = pd.DataFrame(history)

plt.figure(figsize=(7, 4))
plt.plot(
    history_df["epoch"],
    history_df["train_loss"],
    marker="o",
)
plt.xlabel("Epoch")
plt.ylabel("BCE loss")
plt.title("Neural Ranker Training Loss")
plt.show()

## 7. Run batched inference on test data

In [ ]:
def predict_in_batches(model, X, batch_size=4096):
    model.eval()
    scores = []

    with torch.inference_mode():
        for start in range(0, len(X), batch_size):
            batch = torch.from_numpy(
                X[start:start + batch_size]
            ).to(DEVICE)

            logits = model(batch)
            probs = torch.sigmoid(logits)

            scores.append(
                probs.cpu().numpy()
            )

    return np.concatenate(scores)


start = perf_counter()
neural_scores = predict_in_batches(
    model,
    X_test,
)
elapsed = perf_counter() - start

print(f"Inference time: {elapsed:.4f} seconds")
print(f"Throughput: {len(X_test) / elapsed:,.0f} rows/sec")

In [ ]:
print(
    "ROC-AUC:",
    f"{roc_auc_score(y_test, neural_scores):.4f}",
)
print(
    "Average Precision:",
    f"{average_precision_score(y_test, neural_scores):.4f}",
)

## 8. Evaluate ranking quality

In [ ]:
def ranking_metrics(frame, score_col, ks=(5, 10, 20)):
    ranked = frame.sort_values(
        ["user_id", score_col],
        ascending=[True, False],
    )

    results = []

    for user_id, group in ranked.groupby("user_id", sort=False):
        labels = group["label"].to_numpy()
        positives = labels.sum()

        if positives == 0:
            continue

        row = {"user_id": user_id}

        pos = np.flatnonzero(labels == 1)
        row["MRR"] = 1.0 / (pos[0] + 1)

        for k in ks:
            top = labels[:k]
            row[f"Recall@{k}"] = top.sum() / positives

            discounts = 1.0 / np.log2(
                np.arange(2, len(top) + 2)
            )
            dcg = (top * discounts).sum()

            ideal_len = min(int(positives), k)
            idcg = (
                1.0 / np.log2(
                    np.arange(2, ideal_len + 2)
                )
            ).sum()

            row[f"NDCG@{k}"] = dcg / idcg

        results.append(row)

    per_user = pd.DataFrame(results)
    return per_user.mean(numeric_only=True)


predictions = test[
    ["user_id", "streamer", "label", "seen_before"]
].copy()

predictions["score_neural"] = neural_scores

ranking_metrics(
    predictions,
    "score_neural",
)

## 9. Save neural ranker artifacts

In [ ]:
torch.save(
    {
        "state_dict": model.state_dict(),
        "input_dim": len(FEATURE_COLUMNS),
        "feature_columns": FEATURE_COLUMNS,
    },
    ARTIFACT_DIR / "neural_ranker.pt",
)

joblib.dump(
    {
        "imputer": imputer,
        "scaler": scaler,
        "feature_columns": FEATURE_COLUMNS,
    },
    ARTIFACT_DIR / "neural_ranker_preprocessing.joblib",
)

predictions.to_csv(
    PRED_DIR / "neural_predictions.csv.gz",
    index=False,
    compression="gzip",
)

history_df.to_csv(
    ARTIFACT_DIR / "neural_training_history.csv",
    index=False,
)

print("Saved neural model, preprocessing, predictions, and history.")

## 10. Optional: save a TorchScript version for later inference benchmarks

In [ ]:
model.eval()

example_input = torch.zeros(
    1,
    len(FEATURE_COLUMNS),
    dtype=torch.float32,
)

traced_model = torch.jit.trace(
    model.cpu(),
    example_input,
)

traced_model.save(
    str(ARTIFACT_DIR / "neural_ranker_traced.pt")
)

print("Saved TorchScript model.")